In [ ]:
import cv2
import numpy as np

#  config 
CAMERA_INDEX  = 0          # change to 1, 2 … if your webcam isn't index 0
RATIO         = 0.75       # Lowe's ratio test threshold
MAX_FEATURES  = 600

# feature tools 
sift  = cv2.SIFT_create(nfeatures=MAX_FEATURES)
flann = cv2.FlannBasedMatcher({"algorithm": 1, "trees": 5}, {"checks": 50})

def get_features(img):
    return sift.detectAndCompute(img, None)

def draw_keypoints(img, kps):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawKeypoints(vis, kps, vis, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return vis

def match_features(d1, d2):
    if d1 is None or d2 is None or len(d1) < 2 or len(d2) < 2:
        return []
    raw = flann.knnMatch(d1, d2, k=2)
    return [m for m, n in raw if len([m, n]) == 2 and m.distance < RATIO * n.distance]

def draw_flow(img, kps1, kps2, matches):
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    pts1 = np.array([kps1[m.queryIdx].pt for m in matches], dtype=np.float32)
    pts2 = np.array([kps2[m.trainIdx].pt for m in matches], dtype=np.float32)
    for (x0, y0), (x1, y1) in zip(pts1.astype(int), pts2.astype(int)):
        cv2.arrowedLine(vis, (x0, y0), (x1, y1), (0, 255, 0), 1, tipLength=0.3)
        cv2.circle(vis, (x0, y0), 2, (0, 0, 255), -1)
    return vis

# main loop
def run():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("Could not open camera.")
        return

    prev_gray, prev_kps, prev_desc = None, None, None
    frame_idx = 0

    print("Press Q to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        kps, desc = get_features(gray)

        feat_vis = draw_keypoints(gray, kps)
        cv2.putText(feat_vis, f"Frame {frame_idx}  |  {len(kps)} kpts",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 1)
        cv2.imshow("SIFT Features", feat_vis)

        if prev_kps is not None:
            matches = match_features(prev_desc, desc)
            match_vis = cv2.drawMatches(
                prev_gray, prev_kps, gray, kps, matches[:100], None,
                matchColor=(0, 255, 0),
                flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
            )
            cv2.putText(match_vis, f"{len(matches)} matches",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 1)
            cv2.imshow("SIFT Matches", match_vis)

            if matches:
                flow_vis = draw_flow(gray, prev_kps, kps, matches)
                cv2.putText(flow_vis, f"Optical Flow  |  {len(matches)} vectors",
                            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 1)
                cv2.imshow("Optical Flow", flow_vis)

            print(f"[{frame_idx:05d}]  {len(kps)} kpts  |  {len(matches)} matches")

        prev_gray, prev_kps, prev_desc = gray, kps, desc
        frame_idx += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run()

Press Q to quit.
[00001]  600 kpts  |  450 matches
[00002]  600 kpts  |  383 matches
[00003]  600 kpts  |  412 matches
[00004]  600 kpts  |  454 matches
[00005]  600 kpts  |  388 matches
[00006]  600 kpts  |  419 matches
[00007]  600 kpts  |  388 matches
[00008]  600 kpts  |  445 matches
[00009]  600 kpts  |  392 matches
[00010]  601 kpts  |  442 matches
[00011]  600 kpts  |  435 matches
[00012]  600 kpts  |  401 matches
[00013]  601 kpts  |  450 matches
[00014]  600 kpts  |  400 matches
[00015]  600 kpts  |  453 matches
[00016]  600 kpts  |  437 matches
[00017]  600 kpts  |  367 matches
[00018]  600 kpts  |  365 matches
[00019]  600 kpts  |  303 matches
[00020]  600 kpts  |  390 matches
[00021]  600 kpts  |  396 matches
[00022]  600 kpts  |  339 matches
[00023]  601 kpts  |  406 matches
[00024]  600 kpts  |  337 matches
[00025]  600 kpts  |  393 matches
[00026]  600 kpts  |  372 matches
[00027]  601 kpts  |  285 matches
[00028]  600 kpts  |  416 matches
[00029]  600 kpts  |  425 match